# MB-OSA and Adaptive Slice-Width Mapping
This notebook analyzes native 1/2/4-bit MB-OSA simulations and the 8-bit analog reference. Total input and weight precision remains 8 bits. Accuracy is **NOT_MODELED** and is never used for selection.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
run_dir = Path(os.environ['OPTICALLOOP_MULTISLICE_RUN_DIR']).resolve()
artifacts = run_dir / 'artifacts-multislice'
metadata = json.loads((run_dir / 'run.json').read_text())
fixed = pd.read_csv(artifacts / 'fixed_width_summary.csv')
aswm = pd.read_csv(artifacts / 'aswm_summary.csv')
selections = pd.read_csv(artifacts / 'aswm_layer_selections.csv')
checks = pd.read_csv(artifacts / 'validation.csv')
metadata['tier'], metadata['successful_jobs'], metadata['expected_jobs']

## Reproduced configuration and coverage

In [ ]:
manifest = metadata['manifest']
configuration = pd.DataFrame(manifest['architectures']).set_index('name')
coverage = fixed.groupby(['network', 'variant', 'architecture']).layers.max().unstack('variant')
display(configuration)
display(coverage)
assert metadata['successful_jobs'] == metadata['expected_jobs']
assert checks.loc[checks.status == 'FAIL'].empty

## Validation and accuracy boundary

In [ ]:
display(checks)
assert fixed.accuracy_status.eq('NOT_MODELED').all()
assert aswm.accuracy_status.eq('NOT_MODELED').all()

## Primary 5.2 pJ/bit model

In [ ]:
primary_fixed = fixed[(fixed.energy_model == 'linear_bit') & (fixed.optical_loss_db_per_stage == 0)]
primary_aswm = aswm[(aswm.energy_model == 'linear_bit') & (aswm.optical_loss_db_per_stage == 0)]
best_fixed = primary_fixed[primary_fixed.slice_bits.isin([1, 2, 4])].sort_values('edp_j_s').groupby(['network', 'architecture'], as_index=False).first()
headline = primary_aswm.merge(best_fixed[['network', 'architecture', 'slice_bits', 'edp_j_s']], on=['network', 'architecture'], suffixes=('_aswm', '_fixed'))
headline['edp_reduction_vs_best_fixed'] = 1 - headline.edp_j_s_aswm / headline.edp_j_s_fixed
display(headline.sort_values(['network', 'edp_j_s_aswm']))
display(primary_fixed.sort_values(['network', 'edp_j_s']))

## Energy-delay and slice selections

In [ ]:
ax = primary_fixed.plot.scatter(x='latency_s', y='energy_j', c='slice_bits', colormap='viridis', logx=True, logy=True, figsize=(8, 6))
ax.set_title('MB-OSA fixed-width energy-delay points')
plt.show()
selection_counts = selections[(selections.energy_model == 'linear_bit') & (selections.optical_loss_db_per_stage == 0)].groupby(['network', 'slice_bits']).size().unstack(fill_value=0)
display(selection_counts)
selection_counts.plot.bar(stacked=True, figsize=(10, 5), title='ASWM selected slice widths')
plt.show()

## Sensitivity boundary
The optimistic constant-symbol and conservative Walden DAC models, together with 0/0.5/1 dB optical-loss compensation per delay stage, are modeling sensitivities—not measured accuracy results.

In [ ]:
sensitivity = aswm.groupby(['energy_model', 'optical_loss_db_per_stage']).edp_j_s.agg(['min', 'median', 'max'])
display(sensitivity)